# Extra check 2 - Direct speech path: what does the hidden special token replace?

In the text-only test (extra check 1) the model sometimes produced its real special unknown token (ID 1). Our decoding step uses `skip_special_tokens=True`, so that token is removed and no `<unk>` is visible. The earlier `<unk>` investigation covered 35 clips with a visible `<unk>`; the other clips were not checked.

Here we re-run the Direct speech model on a balanced sample of clips (same model, same pinned revision, one clip at a time, greedy, `max_new_tokens=256`, target `cmn`, FP16), keep the token IDs, count ID 1, and check that the re-run reproduces the original outputs. It is a post-hoc check; the original 4,200 predictions are not changed.

Put these files in one Google Drive folder (default `MyDrive/cs760`): `merged_3system_with_metrics.csv`, `final_sample_600_per_group.csv`, `final_sample_audio.zip` (about 150 MB) and, optionally, `text_to_text_predictions.csv` (from extra check 1). Results are saved to a subfolder of it every 25 clips, so a disconnected run can be resumed; the GPU is released at the end.

In [ ]:
%pip install -q "transformers>=4.40,<5" sentencepiece protobuf sacrebleu statsmodels pandas tqdm librosa

In [ ]:
import os, json, re, time, zipfile
from pathlib import Path

USE_DRIVE = True                                   # keep inputs + results on Google Drive (recommended)
DRIVE_DIR = "/content/drive/MyDrive/cs760"         # folder with the input files; results go in a subfolder
HF_CACHE_ON_DRIVE = False                          # True = keep the downloaded model on Drive (needs ~10 GB free Drive space)

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = Path(DRIVE_DIR)
    if HF_CACHE_ON_DRIVE:
        os.environ["HF_HOME"] = str(DATA_DIR / "hf_cache")   # must be set before transformers is imported
else:
    DATA_DIR = Path(".")

import numpy as np, pandas as pd, torch, torchaudio, transformers, sacrebleu
from tqdm.auto import tqdm
from transformers import AutoProcessor, SeamlessM4Tv2ForSpeechToText

MODEL_ID = "facebook/seamless-m4t-v2-large"
MODEL_REVISION = "5f8cc790b19fc3f67a61c105133b20b34e3dcb76"   # same as the main Direct run
TARGET_LANGUAGE = "cmn"
GENERATION_KWARGS = {"num_beams": 1, "do_sample": False, "max_new_tokens": 256}   # same as main run

MERGED_CSV = DATA_DIR / "merged_3system_with_metrics.csv"
METADATA_CSV = DATA_DIR / "final_sample_600_per_group.csv"
AUDIO_ZIP = DATA_DIR / "final_sample_audio.zip"
TEXT_RUN_CSV = DATA_DIR / "text_to_text_predictions.csv"    # optional
AUDIO_DIR = Path("/content/final_sample_audio")             # local disk (fast); only the needed clips are unzipped

MAX_CLIPS = 700               # None = all 4,200 (about 65 minutes of GPU time)
MAX_MINUTES = 20              # stop the inference loop after this many minutes
SANITY_CLIPS = 8              # check reproduction after this many clips
SEED = 760
AUTO_RELEASE_RUNTIME = True   # give the GPU back at the end (only if USE_DRIVE is True)

OUT_DIR = DATA_DIR / "special_unk_check"      # fixed name so a re-run can resume
OUT_DIR.mkdir(parents=True, exist_ok=True)
RESULT_CSV = OUT_DIR / "speech_special_unk_check.csv"

CUDA = torch.cuda.is_available()
DEVICE = torch.device("cuda" if CUDA else "cpu")
DTYPE = torch.float16 if CUDA else torch.float32
print("device:", DEVICE, "| dtype:", DTYPE, "| transformers:", transformers.__version__, "| torch:", torch.__version__)
if not CUDA:
    print("WARNING: no GPU. This will be very slow and not identical to the T4/FP16 run.")

In [ ]:
from huggingface_hub import login
login()   # paste your own Hugging Face token when asked; do not commit it

In [ ]:
merged = pd.read_csv(MERGED_CSV)
meta = pd.read_csv(METADATA_CSV)
assert len(merged) == len(meta) == 4200
# rows must line up one-to-one (same order as the main run)
assert (merged.sentence.values == meta.sentence.values).all() and (merged.primary_accent.values == meta.primary_accent.values).all()
assert merged.id.tolist() == [f"sample_{i:04d}" for i in range(1, 4201)]
merged["clip"] = meta["clip"].values

# Shuffled order that rotates through the 7 accents -> every prefix is (almost) balanced.
shuf = merged.sample(frac=1, random_state=SEED)
shuf["_rank"] = shuf.groupby("primary_accent").cumcount()
order = shuf.sort_values("_rank", kind="stable").drop(columns="_rank")
todo = order if MAX_CLIPS is None else order.head(MAX_CLIPS)

# resume: skip clips that are already finished (saved on Drive)
rows = pd.read_csv(RESULT_CSV).to_dict("records") if RESULT_CSV.exists() else []
done_ids = {r["id"] for r in rows}
todo_left = todo[~todo.id.isin(done_ids)]
print(f"target clips: {len(todo)} | already done: {len(done_ids)} | still to run: {len(todo_left)}")

# unzip only the clips we still need
AUDIO_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(AUDIO_ZIP) as z:
    names = set(z.namelist())
    for c in todo_left["clip"]:
        if not (AUDIO_DIR / c).exists():
            z.extract(f"final_sample_audio/{c}", "/content")
assert all((AUDIO_DIR / c).is_file() for c in todo_left["clip"]), "some audio files are missing"

In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
model = SeamlessM4Tv2ForSpeechToText.from_pretrained(MODEL_ID, revision=MODEL_REVISION, dtype=DTYPE)
model.to(DEVICE).eval()
SR = int(getattr(processor.feature_extractor, "sampling_rate", 16000))
UNK_ID = processor.tokenizer.unk_token_id
print("sample rate:", SR, "| unk_token_id:", UNK_ID)

In [ ]:
LOADER_USED = {"torchaudio": 0, "librosa": 0}
def load_audio(path):
    # same steps as the main run: mono (mean), resample, float32.
    # If torchaudio cannot decode mp3 on this machine, fall back to librosa (and count it).
    try:
        wav, sr = torchaudio.load(str(path))
        wav = wav.mean(dim=0)
        if sr != SR:
            wav = torchaudio.functional.resample(wav, sr, SR)
        LOADER_USED["torchaudio"] += 1
        return wav.to(torch.float32).contiguous().cpu().numpy()
    except Exception:
        import librosa
        wav, _ = librosa.load(str(path), sr=SR, mono=True)
        LOADER_USED["librosa"] += 1
        return wav.astype(np.float32)

MARK = "[UNKID]"
def decode_variants(ids):
    """(a) original style: skip special tokens (ID 1 vanishes).  (b) same, but ID 1 replaced by a visible marker."""
    a = processor.decode(ids, skip_special_tokens=True).strip()
    parts, cur = [], []
    for t in ids:
        if t == UNK_ID:
            parts.append(processor.decode(cur, skip_special_tokens=True)); cur = []
        else:
            cur.append(t)
    parts.append(processor.decode(cur, skip_special_tokens=True))
    return a, MARK.join(parts).strip()

def run_one(path):
    inputs = processor(audio=load_audio(path), sampling_rate=SR, return_tensors="pt")
    mi = {k: (v.to(DEVICE, dtype=DTYPE) if torch.is_floating_point(v) else v.to(DEVICE)) for k, v in inputs.items()}
    with torch.inference_mode():
        gen = model.generate(**mi, tgt_lang=TARGET_LANGUAGE, **GENERATION_KWARGS)
    ids = gen[0]
    if ids.ndim > 1:
        ids = ids[0]
    ids = ids.detach().cpu().tolist()
    a, b = decode_variants(ids)
    pos = [i for i, t in enumerate(ids) if t == UNK_ID]
    return dict(rerun_translation=a, rerun_translation_marked=b, n_special_unk_ids=len(pos),
                special_unk_positions=json.dumps(pos), n_tokens=len(ids))

def save():
    pd.DataFrame(rows).to_csv(RESULT_CSV, index=False)

if len(todo_left):
    _ = run_one(AUDIO_DIR / todo_left["clip"].iloc[0])      # warm-up (not saved)
t0 = time.time()
stopped_by = "finished all target clips"
pbar = tqdm(total=len(todo_left))
for k, r in enumerate(todo_left.itertuples(), start=1):
    out = run_one(AUDIO_DIR / r.clip)
    rows.append(dict(id=r.id, primary_accent=r.primary_accent, sentence=r.sentence,
                     reference_translation_zh=r.reference_translation_zh, original_translation=r.direct_translation, **out))
    pbar.update(1)
    if k % 25 == 0:
        save()
    if k == SANITY_CLIPS and not done_ids:                # first run only: sanity gate
        same8 = np.mean([x["rerun_translation"] == x["original_translation"] for x in rows[:SANITY_CLIPS]])
        print(f"\nSanity check after {SANITY_CLIPS} clips: re-run identical to the original in {same8*100:.0f}%")
        if same8 < 0.5:
            save(); stopped_by = "sanity gate failed"
            print("STOP: the re-run does not match the original outputs. Check audio loader / library versions before spending more GPU time.")
            break
    if (time.time() - t0) / 60 > MAX_MINUTES:
        stopped_by = f"time budget reached ({MAX_MINUTES} min)"
        break
pbar.close(); save()
res = pd.DataFrame(rows)
print(f"\nclips finished in total: {len(res)} | stopped by: {stopped_by} | this session: {(time.time()-t0)/60:.1f} min | audio loader used: {LOADER_USED}")

In [ ]:
# Free the GPU memory now (the analysis below needs no GPU)
del model
import gc; gc.collect()
if CUDA:
    torch.cuda.empty_cache()

In [ ]:
from statsmodels.stats.proportion import proportion_confint
UNK_RE = re.compile(r"<\s*unk\s*>", re.IGNORECASE)
for c in ["rerun_translation", "original_translation", "rerun_translation_marked"]:
    res[c] = res[c].fillna("").astype(str)      # safe after resuming from the saved CSV
res["visible_unk"] = res.rerun_translation.map(lambda s: bool(UNK_RE.search(s)))
res["orig_visible_unk"] = res.original_translation.map(lambda s: bool(UNK_RE.search(str(s))))
res["special_unk"] = res.n_special_unk_ids > 0
n = len(res)

# 0) Did the re-run reproduce the original outputs?
same = (res.rerun_translation == res.original_translation)
print(f"Re-run output identical to the original: {same.mean()*100:.1f}%  ({same.sum()}/{n})")
print(f"Visible <unk> agreement with the original: {(res.visible_unk == res.orig_visible_unk).mean()*100:.1f}%")

# 1) Rates
def line(name, k):
    lo, hi = proportion_confint(k, n, method="wilson")
    print(f"{name:55s} {k:4d}/{n}  {100*k/n:5.1f}%   95% CI [{100*lo:.1f}, {100*hi:.1f}]")
line("Visible <unk> (spelled out; what we counted: 508 of 4,200)", int(res.visible_unk.sum()))
line("Special UNK token ID 1 generated (silently removed)", int(res.special_unk.sum()))
print("total ID-1 tokens:", int(res.n_special_unk_ids.sum()))

# 2) Are they the same clips?
print()
print(pd.crosstab(res.visible_unk, res.special_unk, rownames=["visible <unk>"], colnames=["special ID 1"]))

In [ ]:
# 3) By accent (small numbers per accent if you stopped early; read with care)
(res.groupby("primary_accent")[["visible_unk", "special_unk"]].mean() * 100).round(1).assign(clips=res.groupby("primary_accent").size())

In [ ]:
# 4) Quality of these clips (different clips, so not a measure of effect) (sentence chrF++, same settings as the team's CSV)
chrf = sacrebleu.metrics.CHRF(word_order=2)
sc = lambda h, r: chrf.sentence_score(h if isinstance(h, str) else "", [r]).score
res["chrfpp"] = [sc(h, r) for h, r in zip(res.rerun_translation, res.reference_translation_zh)]
res["group"] = np.select([res.visible_unk, res.special_unk], ["visible <unk>", "special ID 1 only (silent)"], "neither")
print(res.groupby("group").chrfpp.agg(["mean", "size"]).round(2))

In [ ]:
# 5) Examples: where the hidden token sits
silent = res[res.special_unk & ~res.visible_unk].head(15)
for r in silent.itertuples():
    print("EN   :", r.sentence); print("MODEL:", r.rerun_translation_marked); print("REF  :", r.reference_translation_zh); print()

In [ ]:
# 6) Optional: compare with the text-only run (extra check 1)
if TEXT_RUN_CSV.exists():
    t = pd.read_csv(TEXT_RUN_CSV)[["id", "text_ref_special_unk_ids"]]
    m = res.merge(t, on="id")
    m["text_special"] = m.text_ref_special_unk_ids > 0
    print(pd.crosstab(m.special_unk, m.text_special, rownames=["speech: special ID 1"], colnames=["text: special ID 1"]))
else:
    print("text_to_text_predictions.csv not found; skipping.")

In [ ]:
lo, hi = proportion_confint(int(res.special_unk.sum()), n, method="wilson")
summary = {
    "model_id": MODEL_ID, "revision": MODEL_REVISION, "device": str(DEVICE), "dtype": str(DTYPE),
    "generation": GENERATION_KWARGS, "clips_run": int(n), "max_clips": MAX_CLIPS, "seed": SEED,
    "stopped_by": stopped_by, "audio_loader_counts": LOADER_USED,
    "rerun_identical_to_original_pct": float(same.mean() * 100),
    "visible_unk_clips": int(res.visible_unk.sum()),
    "special_unk_id_clips": int(res.special_unk.sum()),
    "special_unk_id_rate_pct": float(100 * res.special_unk.mean()),
    "special_unk_id_rate_ci95_pct": [float(100 * lo), float(100 * hi)],
    "special_and_visible_clips": int((res.special_unk & res.visible_unk).sum()),
    "note": "Post-hoc diagnostic. Original 4,200 Direct predictions unchanged.",
}
(OUT_DIR / "run_summary_unk_extra_2_speech.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))
print("\nResults saved to:", OUT_DIR)

In [ ]:
# release the GPU; the results are already on Google Drive
if USE_DRIVE and AUTO_RELEASE_RUNTIME:
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as e:
        print("Could not release automatically:", e, "\nUse Runtime -> Disconnect and delete runtime.")
else:
    print("Remember: Runtime -> Disconnect and delete runtime, to stop using GPU quota.")